# EDA básico e inspección de calidad ECYL

Lo primero es importar las librerías que vamos a usar.

In [2]:
import requests
import os
import pandas as pd
import json
from pathlib import Path
import matplotlib.pyplot as plt

In [3]:
api_key = os.environ.get("API_KEY")
headers = {
    "Authorization": f"Apikey {api_key}",
    "Content-Type": "application/json"
}

Prueba de que se ha podido acceder a la API

In [4]:
url = "https://analisis.datosabiertos.jcyl.es/api/explore/v2.1/catalog/datasets/ofertas-de-empleo/records/"

limite = 100
offset = 0
todos_los_registros = []
total = None

while total is None or offset < total:
    respuesta = requests.get(
        url,
        params={"lang": "es", "limit": limite, "offset": offset},
        timeout=30
    )
    respuesta.raise_for_status()

    pagina = respuesta.json()
    total = pagina["total_count"]
    registros = pagina["results"]

    if not registros:
        break

    todos_los_registros.extend(registros)
    offset += len(registros)

print(f"Descargados {len(todos_los_registros)} de {total} registros")

with open("respuestas.json", "w", encoding="utf-8") as archivo:
    json.dump(
        {"total_count": total, "results": todos_los_registros},
        archivo,
        ensure_ascii=False,
        indent=2
    )

Descargados 1027 de 1027 registros


# Carga de datos

Mostramos que se han obtenido correctamente

In [5]:
with open("respuestas.json", "r", encoding="utf-8") as archivo:
    datos = json.load(archivo)

df = pd.json_normalize(datos["results"])

print(df.shape)  # filas y columnas
df.head()

(1027, 16)


,titulo,provincia,fecha_publicacion,descripcion,provincia_alternativa,fuente_contenido,localidad_asset,localidad_asset0,localidad_asset2,documentos_asociados,identificador,actualizacionmetadatos,enlace_al_contenido,geografico.lon,geografico.lat,geografico
0,"COCINEROS, EN GENERAL",Palencia,2026-10-12,"<p>COCINEROS, PARA DUEÑAS. LABORAL INDEFINIDO ...",NaN,ECYL,1179298081851,Dueñas,34.0,None,1285689059800,2026-10-12,https://datosabiertos.jcyl.es/web/jcyl/Empleo/...,-4.546755,41.877316,NaN
1,"COCINEROS, EN GENERAL",Soria,2026-10-11,<p>OFERTA 21165. HOTEL EN EL BURGO DE OSMA (SO...,NaN,ECYL,1179469126353,Burgo de Osma-Ciudad de Osma,42.0,None,1285689059772,2026-10-12,https://datosabiertos.jcyl.es/web/jcyl/Empleo/...,-3.075403,41.588287,NaN
2,TERAPEUTAS OCUPACIONALES,León,2026-10-11,"<p>Para más información acuda a <a href=""http:...",NaN,ECYL,1179297955047,Soto de la Vega,24.0,None,1285689059856,2026-10-12,https://datosabiertos.jcyl.es/web/jcyl/Empleo/...,-5.882215,42.332270,NaN
3,"PEONES DE LA INDUSTRIA DE LA ALIMENTACIÓN, BEB...",Salamanca,2026-10-10,"<p>Para más información acuda a <a href=""http:...",NaN,ECYL,1179298266466,Ledrada,37.0,None,1285689059884,2026-10-12,https://datosabiertos.jcyl.es/web/jcyl/Empleo/...,-5.720769,40.469257,NaN
4,Técnico/a Medio de Contabilidad para Diputació...,Palencia,2026-10-09,<p><strong>Convoca: </strong> Diputación de Pa...,NaN,Organismo oficial,NaN,NaN,NaN,None,1285688752400,2026-10-09,https://datosabiertos.jcyl.es/web/jcyl/Empleo/...,NaN,NaN,NaN


# Estudio de la muestra

Mostramos la dimensión del dataset

In [6]:
registros = df.shape[0]
columnas = df.shape[1]
print(f"El DataFrame tiene {registros} registros y {columnas} columnas.")

El DataFrame tiene 1027 registros y 16 columnas.


Comprobamos de que tipo son las columnas del dataset

In [7]:
for columna in df.columns:
    print(f"{columna}: {df[columna].dtype}")

titulo: str
provincia: str
fecha_publicacion: str
descripcion: str
provincia_alternativa: str
fuente_contenido: str
localidad_asset: str
localidad_asset0: str
localidad_asset2: float64
documentos_asociados: object
identificador: str
actualizacionmetadatos: str
enlace_al_contenido: str
geografico.lon: float64
geografico.lat: float64
geografico: float64


Podemos ver que la mayoría de columnas son cadenas de texto salvo localidad_asset2, geografico.lon, geografico.lat y geografico que son númericas. Documentos asociados es un object que también funciona como una cadena de texto.

# Variables principales

La primera columna es el título

In [8]:
print("Número de valores únicos en la columna título:", df["titulo"].nunique())
print(df["titulo"].unique())

Número de valores únicos en la columna título: 441
<StringArray>
[                                                 'COCINEROS, EN GENERAL',
                                               'TERAPEUTAS OCUPACIONALES',
            'PEONES DE LA INDUSTRIA DE LA ALIMENTACIÓN, BEBIDAS Y TABACO',
            'Técnico/a Medio de Contabilidad para Diputación de Palencia',
             'Alguacil/a para Ayto. de Villamoratiel de las Matas (León)',
                                                      'PINCHES DE COCINA',
                                                  'CAMAREROS, EN GENERAL',
                                 'PODADORES Y/O MOTOSERRISTAS FORESTALES',
                                            'FISIOTERAPEUTAS, EN GENERAL',
              'MOZOS DE CARGA Y DESCARGA, ALMACÉN Y/O MERCADO DE ABASTOS',
 ...
                               'Técnico/a de Mantenimiento para Palencia',
          'Condutor/a repartidor/a para Villares de la Reina (Salamanca)',
                          'Téc

Como podemos observar, de los 1027 registros solo existen 441 valores únicos en la columna de títulos. En este dataset es la columna que le da nombre a la oferta de empleo. 

Las columnas de provincia y provincia alternativa contienen los lugares donde se van a llevar a cabo los trabajos de las ofertas. 

Las columnas de localidad_asset son columnas que continen el municipio en el que se va a realizar el trabajo. La primera es un identificador, la segunda es el nombre del municipio y la tercera contiene otro número asociado con la localidad.

In [9]:
print("Numero de valores únicos en la columna descripcion:", df["descripcion"].nunique())

Numero de valores únicos en la columna descripcion: 1024


Como podemos ver la columna de descripciones no tiene casi valores repetidos. Esta columna tiene la función de dar detalles sobre el empleo de la oferta. 

In [10]:
print("Numero de valores únicos en la columna fecha_publicacion:", df["fecha_publicacion"].nunique())

Numero de valores únicos en la columna fecha_publicacion: 127


La columna de fecha_publicacion es el lugar donde se data la fecha en la que se realizó la publicación de la oferta de empleo. Las fechas coinciden en muchas ofertas de empleo.

In [11]:
print("Número de valores únicos en la columna documentos_asociados:", df["documentos_asociados"].nunique())

Número de valores únicos en la columna documentos_asociados: 0


La columna de documentos asociados está vacía. Es la columna donde se encontrarían los documentos que estarían asociados a las diferentes ofertas.

Las columnas geográficas contienen los puntos geográficos de los lugares donde se realizaran los trabajos. Están divididas en la columnade latitud, de longitud y el punto completo.

In [12]:
print("Número de valores únicos en la columna identificador:", df["identificador"].nunique())
print(df["identificador"].unique())

Número de valores únicos en la columna identificador: 1027
<StringArray>
['1285689059800', '1285689059772', '1285689059856', '1285689059884',
 '1285688752400', '1285688741007', '1285688777604', '1285688636501',
 '1285688636081', '1285688593879',
 ...
 '1285701423011', '1285701423095', '1285701411818', '1285701271826',
 '1285701412123', '1285701232857', '1285701412253', '1285701271891',
 '1285701333113', '1285701333503']
Length: 1027, dtype: str


La columna de identificador contiene un número que identifica cada oferta de empleo. Es la columna que no tiene ningú valor repetido. La columna clave para diferenciar los registros. Está en formato de cadena de texto.

In [13]:
print("Número de valores únicos en la columna fuente_contenido:", df["fuente_contenido"].nunique())
print(df["fuente_contenido"].unique())

Número de valores únicos en la columna fuente_contenido: 4
<StringArray>
['ECYL', 'Organismo oficial', 'Red EURES', 'Internet / Prensa']
Length: 4, dtype: str


La columna de fuente de contenido nos indica la procedencia de la oferta de empleo. Solo tiene cuatro valores únicos.